In [11]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
from mpi4py import MPI
import pickle
# pwd = os.getcwd()


# torch
import torch

# quimb
import quimb.tensor as qtn
import autoray as ar

from vmc_torch.experiment.tn_model import *
from vmc_torch.experiment.tn_model import init_weights_to_zero, init_weights_uniform
from vmc_torch.sampler import MetropolisExchangeSamplerSpinful, MetropolisExchangeSamplerSpinful_2D_reusable
from vmc_torch.variational_state import Variational_State
from vmc_torch.optimizer import SGD, SR, DecayScheduler
from vmc_torch.VMC import VMC
from vmc_torch.hamiltonian_torch import spinful_Fermi_Hubbard_square_lattice_torch
from vmc_torch.torch_utils import SVD,QR,QR_tao
from vmc_torch.fermion_utils import unpack_ftns
from vmc_torch.utils import closest_divisible

# Register safe SVD and QR functions to torch
ar.register_function('torch','linalg.svd',SVD.apply)
ar.register_function('torch','linalg.qr',QR_tao.apply)
pwd = '/home/sijingdu/TNVMC/VMC_code/vmc_torch/scripts/4x4/test_data'

COMM = MPI.COMM_WORLD
SIZE = COMM.Get_size()
RANK = COMM.Get_rank()

# Hamiltonian parameters
Lx = int(4)
Ly = int(4)
symmetry = 'U1SU'
t = 1.0
U = 8.0
N_f = int(Lx*Ly-2)
n_fermions_per_spin = (N_f//2, N_f//2)
H = spinful_Fermi_Hubbard_square_lattice_torch(Lx, Ly, t, U, N_f, pbc=False, n_fermions_per_spin=n_fermions_per_spin)
graph = H.graph
# TN parameters
D = 4
chi = 8
dtype=torch.float64

if symmetry == 'U1SU':
    su_skeleton = 'peps_skeleton_U1SU.pkl'
    su_params = 'peps_su_params_U1SU.pkl'
    symmetry = 'Z2'
else:
    su_skeleton = 'peps_skeleton.pkl'
    su_params = 'peps_su_params.pkl'
    
# Load PEPS
peps = unpack_ftns(
    params_path=pwd+f"/{Lx}x{Ly}/t={t}_U={U}/N={N_f}/{symmetry}/D={D}/{su_params}",
    skeleton_path=pwd+f"/{Lx}x{Ly}/t={t}_U={U}/N={N_f}/{symmetry}/D={D}/{su_skeleton}",
    scale=1.0,
    new_symmray_format=True,
)

# VMC sample size
N_samples = int(20)
N_samples = closest_divisible(N_samples, SIZE)
if (N_samples/SIZE)%2 != 0:
    N_samples += SIZE

# Set up variational model
contraction_kwargs = {
    "mode": "fit",
    "bsz": 2,
    "max_iterations": 50,
    "tn_fit": "zipup",
    "progbar": False,
    "tol": 1e-5,
}
contraction_kwargs = {"mode": "mps"}
model = fTN_backflow_attn_Tensorwise_Model_v1(
    peps,
    max_bond=chi,
    embedding_dim=16,
    attention_heads=4,
    nn_final_dim=D,
    nn_eta=1.0,
    dtype=dtype,
)
nnbf_model = NNBF(
    hilbert=H.hilbert,
    param_dtype=dtype,
    hidden_dim=32,
    nn_eta=1.0,
)

In [12]:
model.num_params, nnbf_model.num_params

(36816, 16288)

In [13]:
fxs = torch.tensor(list(H.hilbert.random_state() for _ in range(10)), dtype=torch.int64)

In [14]:
%%timeit
with torch.no_grad():
    model(fxs)

198 ms ± 3.68 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [15]:
%%timeit
with torch.no_grad():
    nnbf_model(fxs)

2.85 ms ± 21.7 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
